## Attention mechanism for a single query

The major bottleneck for RNNs was that it had to remember the entire encoded input in a single hidden state.

Attention mechanism aims to selectively access different parts of the input sequence at each decoding step. Making it a viable option for longer sequences.

The "self" refers to the mechanism's ability to compute attention weights by assessing relationships and dependencies between the various parts of the input itself.

In [1]:
#i n self attention - goal is to calculate context vector z_i for each input element x_i.
# A context vector can be interpreted as an enriched embedding vector.

In [2]:
import torch

In [3]:
inputs = torch.tensor(
[[0.43, 0.15, 0.89], # Your    (x^1)
 [0.55, 0.87, 0.66], # journey (x^2)
 [0.57, 0.85, 0.64], # starts  (x^3)
 [0.22, 0.58, 0.33], # with    (x^4)
 [0.77, 0.25, 0.10], # one     (x^5)
 [0.05, 0.80, 0.55]] # step    (x^6)
)

In [4]:
# to calculate the intermediate attention scores
# the dot product between each token query and input token

# journey (x^2)
query = inputs[1]
query

tensor([0.5500, 0.8700, 0.6600])

In [5]:
attn_scores_2 = torch.empty(inputs.shape[0])

for i, x_i in enumerate(inputs):
  attn_scores_2[i] = x_i @ query

attn_scores_2

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

In [6]:
# naive normalization
attn_weights_2_tmp = attn_scores_2 /attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


In [7]:
# using softmax

# reasons
# 1. better at extreme values
# 2. offers better gradient properties at training
# 3. supports negative scores
# 4. Ensures that the attention weights are always positive and can be
# interpreted as probabilites



In [8]:
def softmax_naive(x):
  return torch.exp(x)/ torch.exp(x).sum(dim=0)

In [9]:
attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [10]:
# for numerical stability

attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


In [11]:
# to calculate the context vector z^2.
# multiply the attn weights with input tokens
# then sum up


# weighted sum of all input vectors obtained by multiplying each input vector
# with the corresponding attention weight

In [12]:
query = inputs[1]
context_vec_2 = torch.zeros_like(query)

for i, x_i in enumerate(inputs):
  context_vec_2 += x_i * attn_weights_2[i]

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


## Doing this for all input tokens

In [13]:
# steps

# 1. compute attn_scores - dot products between inputs
# 2. compute attn_weights - normalization
# 3. compute context vector - weighted of attn_weights and inputs

In [14]:
attn_scores = inputs @ inputs.T
attn_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [15]:
attn_scores_2, attn_scores[1]

(tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865]),
 tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865]))

In [16]:
attn_weights = torch.softmax(attn_scores, dim=-1)
attn_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [17]:
attn_weights_2, attn_weights[1]

(tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]),
 tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]))

In [18]:
all_context_vecs = attn_weights @ inputs
all_context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

In [19]:
all_context_vecs[1], context_vec_2

(tensor([0.4419, 0.6515, 0.5683]), tensor([0.4419, 0.6515, 0.5683]))

## Implementing Self-Attention with Trainable weights

The terms key, query and value are borrowed from information retrieval and databases

A query is analogous to the search query in a database. It represents the current item the model focuses and tries to understan.

The key is like the database key. In the attention mechanism each itemhas an associated key to match the query

The value represents the actual content or represent of the input items

In [20]:
import torch.nn as nn

In [21]:
class SelfAttention_v1(nn.Module):
  def __init__(self, d_in, d_out, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

  def forward(self, x):
    queries = self.W_query(x)
    keys = self.W_key(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.T
    attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)  #scaling term

    context_vectors = attn_weights @ values
    return context_vectors

The additional scaling term which dividing the attn_scores by the embedding dimension size is to improve the training performance by avoiding small gradients.


As the dot products increase, the softmax function behaves like a step function -> the output is almost like a hot encoded vector. This results in almost zero gradients causing the learning to slow down.

The scaling smooths this out making the output more diffused instead of peaky 1-hot.

In [22]:
torch.manual_seed(789)
sa_v2 = SelfAttention_v1(d_in=3, d_out=2)
print(sa_v2(inputs))

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


## Causal Attention

Also known as masked attention - hiding future words

In [23]:
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs)

attn_scores = queries @ keys.T
attn_scores

tensor([[ 0.2899,  0.0716,  0.0760, -0.0138,  0.1344, -0.0511],
        [ 0.4656,  0.1723,  0.1751,  0.0259,  0.1771,  0.0085],
        [ 0.4594,  0.1703,  0.1731,  0.0259,  0.1745,  0.0090],
        [ 0.2642,  0.1024,  0.1036,  0.0186,  0.0973,  0.0122],
        [ 0.2183,  0.0874,  0.0882,  0.0177,  0.0786,  0.0144],
        [ 0.3408,  0.1270,  0.1290,  0.0198,  0.1290,  0.0078]],
       grad_fn=<MmBackward0>)

In [24]:
context_length = attn_scores.shape[0]

In [25]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
mask

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

In [26]:
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
masked

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)

In [27]:
attn_weights = torch.softmax(masked/keys.shape[-1] ** 0.5, dim=-1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

In [ ]:
# masking additional weights with dropout

In [28]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5)
example = torch.ones(6, 6)
print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [35]:
class CausalAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer(
      'mask',
      torch.triu(torch.ones(context_length, context_length), diagonal=1)
    )

  def forward(self, x):
    b, num_tokens, d_in = x.shape
    queries = self.W_query(x)
    keys = self.W_key(x)
    values = self.W_value(x)

    attn_scores = queries @ keys.transpose(1, 2) # keep batch dimension at 0
    attn_scores.masked_fill_(
        self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
    )
    attn_weights = torch.softmax(attn_scores/keys.shape[-1] ** 0.5, dim=-1)
    attn_weights = self.dropout(attn_weights)

    context_vec = attn_weights @ values
    return context_vec

In [31]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

torch.Size([2, 6, 3])


In [33]:
d_in = 3
d_out = 2

In [37]:
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)
context_vecs = ca(batch)
print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


## Multi-Head attention

In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
    super().__init__()

    assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

    self.d_out = d_out
    self.num_heads = num_heads
    self.head_dim = d_out // num_heads
    self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
    self.out_proj = nn.Linear(d_out, d_out)
    self.dropout = nn.Dropout(dropout)
    self.register_buffer(
        "mask",
        torch.triu(torch.ones(context_length, context_length), diagonal=1)
    )

  def forward(self, x):
    b, num_tokens, d_in = x.shape
    # shapes of the q, k, v -> batch, num_tokens, d_out
    keys = self.W_key(x)
    queries = self.W_query(x)
    values = self.W_value(x)

    # reshape to batch, num_tokens, num_heads, head_dim
    keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
    values = values.view(b, num_tokens, self.num_heads, self.head_dim)
    queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

    keys = keys.transpose(1, 2)
    queries = queries.transpose(1, 2)
    values = values.transpose(1, 2)

    attn_scores = queries @ keys.transpose(2, 3) # dot product for each head

    mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
    attn_scores.masked_fill_(mask_bool, -torch.inf)

    attn_weights = torch.softmax(
        attn_scores / keys.shape[-1] ** 0.5, dim=-1
    )


    context_vec = (attn_weights @ values).transpose(1, 2)

    context_vec = context_vec.continguous().view(
      b, num_tokens, self.d_out
    )

    context_vec = self.out_proj(context_vec)

    return context_vec
